# IM-VRM Grid Analysis and Congestion Mapping
This notebook visualizes the traffic speed profiles, free flow speeds, and hourly congestion levels across binned spatial coordinates.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.utils.helpers import load_config

In [ ]:
config = load_config()
grid_stats_path = os.path.join("..", config["data"]["processed_dir"], "grid_congestion_stats.csv")

if os.path.exists(grid_stats_path):
    grid_stats = pd.read_csv(grid_stats_path)
    print(f"Loaded {len(grid_stats)} spatial-temporal congestion rows.")
    display(grid_stats.head())
else:
    print(f"Grid congestion file not found at {grid_stats_path}. Run `python main.py` first.")

## Spatial Speed Heatmaps
Let's construct a spatial grid of average velocities for specific rush hour slots.

In [ ]:
if 'grid_stats' in locals():
    rows = config["spatial_grid"]["num_rows"]
    cols = config["spatial_grid"]["num_cols"]
    
    # Select morning rush hour (e.g. 8 AM)
    rush_hour_df = grid_stats[grid_stats["hour"] == 8]
    
    speed_matrix = np.full((rows, cols), np.nan)
    congestion_matrix = np.zeros((rows, cols))
    
    for idx, row in rush_hour_df.iterrows():
        r = int(row["grid_row"])
        c = int(row["grid_col"])
        speed_matrix[r, c] = row["avg_speed"]
        congestion_matrix[r, c] = row["congestion_level"]
        
    # Plot
    fig, ax = plt.subplots(1, 2, figsize=(15, 6))
    
    im1 = ax[0].imshow(speed_matrix, origin='lower', cmap='YlOrRd')
    ax[0].set_title('Average Speeds at 8:00 AM (km/h)')
    fig.colorbar(im1, ax=ax[0])
    
    # 0=FreeFlow, 1=Moderate, 2=Congested
    im2 = ax[1].imshow(congestion_matrix, origin='lower', cmap='viridis', vmin=0, vmax=2)
    ax[1].set_title('Congestion Levels at 8:00 AM')
    # Customize colorbar ticks
    cbar = fig.colorbar(im2, ax=ax[1], ticks=[0, 1, 2])
    cbar.ax.set_yticklabels(['FreeFlow', 'Moderate', 'Congested'])
    
    plt.tight_layout()
    plt.show()